# Clustering

UMAP + HDBSCAN + Optuna search on the BGE-M3 embeddings.

In [1]:
import numpy as np
import pandas as pd
import json
import warnings
import umap
import hdbscan
import optuna
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\Jim\miniconda3\envs\diplomatiki2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and normalize

In [2]:
df = pd.read_csv("tovima_embedded.csv", encoding="utf-32", sep="\t")
df["to_lists"] = df["to_lists"].apply(json.loads)
df["to_other_recipients"] = df["to_other_recipients"].apply(json.loads)

embeddings_raw = np.load("tovima_embeddings_bge_m3.npy")

assert len(df) == embeddings_raw.shape[0], "DataFrame and embeddings are not aligned"

# L2 normalize
embeddings = normalize(embeddings_raw, norm="l2")

print(f"Rows: {len(df)}")
print(f"Embedding dim: {embeddings.shape[1]}")
print(f"Norm check (should be ~1.0): {np.linalg.norm(embeddings[0]):.4f}")

Rows: 13637
Embedding dim: 1024
Norm check (should be ~1.0): 1.0000


## 2. PCA pre-reduction

In [3]:
N_PCA = 256

pca = PCA(n_components=N_PCA, svd_solver="randomized", random_state=42)
embeddings_pca = pca.fit_transform(embeddings)

cumvar = np.sum(pca.explained_variance_ratio_)
print(f"PCA: {N_PCA} components retain {cumvar*100:.1f}% of variance")

PCA: 256 components retain 87.7% of variance


## 3. Semantic coherence metric

In [4]:
def semantic_coherence(embeddings, labels, min_cluster_size=10):
    cluster_ids = set(labels) - {-1}
    scores, weights = [], []
    for c in cluster_ids:
        idx = np.where(labels == c)[0]
        if len(idx) < min_cluster_size:
            continue
        vecs = embeddings[idx]
        centroid = vecs.mean(axis=0, keepdims=True)
        sims = cosine_similarity(vecs, centroid).ravel()
        scores.append(np.median(sims))
        weights.append(len(idx))
    if not scores:
        return 0.0
    return float(np.average(scores, weights=weights))

## 4. Pilot run (2,000 rows)

In [5]:
PILOT_N = 2000
rng = np.random.default_rng(42)
pilot_idx = rng.choice(len(embeddings_pca), PILOT_N, replace=False)
pilot_emb = embeddings_pca[pilot_idx]
pilot_emb_orig = embeddings[pilot_idx]

# pilot params
pilot_umap = umap.UMAP(
    n_neighbors=20,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
pilot_reduced = pilot_umap.fit_transform(pilot_emb)

pilot_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)
pilot_labels = pilot_hdbscan.fit_predict(pilot_reduced)

n_clusters = len(set(pilot_labels) - {-1})
noise_pct = float(np.mean(pilot_labels == -1)) * 100
coherence = semantic_coherence(pilot_emb_orig, pilot_labels)
cluster_sizes = sorted([int(np.sum(pilot_labels == c)) for c in set(pilot_labels) - {-1}], reverse=True)

print(f"Pilot ({PILOT_N} rows):")
print(f"  Clusters found:      {n_clusters}")
print(f"  Noise:               {noise_pct:.1f}%")
print(f"  Semantic coherence:  {coherence:.4f}")
print(f"  Largest clusters:    {cluster_sizes[:5]}")
print(f"  Median cluster size: {float(np.median(cluster_sizes)):.0f}")

Pilot (2000 rows):
  Clusters found:      12
  Noise:               5.9%
  Semantic coherence:  0.7470
  Largest clusters:    [550, 478, 183, 141, 90]
  Median cluster size: 82


## 5. Optuna objective

In [6]:
def make_objective(embeddings_pca, embeddings_orig):

    def objective(trial):
        # UMAP params
        n_neighbors = trial.suggest_int("n_neighbors", 10, 50)
        n_components = trial.suggest_int("n_components", 5, 20)
        min_dist = trial.suggest_float("min_dist", 0.0, 0.1)

        # min_cluster_size floor 30
        min_cluster_size = trial.suggest_int("min_cluster_size", 30, 150)
        min_samples = trial.suggest_int(
            "min_samples", 3, max(4, min_cluster_size // 3)
        )

        try:
            reducer = umap.UMAP(
                n_neighbors=n_neighbors,
                n_components=n_components,
                min_dist=min_dist,
                metric="cosine",
                random_state=42,
            )
            reduced = reducer.fit_transform(embeddings_pca)

            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="euclidean",
                cluster_selection_method="eom",
            )
            labels = clusterer.fit_predict(reduced)

        except Exception:
            return -999.0

        n_clusters = len(set(labels) - {-1})

        # hard limits
        if n_clusters < 5 or n_clusters > 200:
            return -999.0

        noise_pct = float(np.mean(labels == -1))
        coherence = semantic_coherence(embeddings_orig, labels)

        # noise penalty
        noise_penalty = max(0.0, noise_pct - 0.15) * 1.0

        # fragmentation penalty
        cluster_sizes = [int(np.sum(labels == c)) for c in set(labels) - {-1}]
        median_size = float(np.median(cluster_sizes))
        frag_penalty = max(0.0, 1.0 - median_size / 50.0) * 0.3

        score = coherence - noise_penalty - frag_penalty

        # log per-trial info
        persistence = clusterer.cluster_persistence_
        if hasattr(persistence, "__len__") and len(persistence) > 0:
            stability = float(np.mean(persistence))
        else:
            stability = 0.0

        trial.set_user_attr("n_clusters", n_clusters)
        trial.set_user_attr("noise_pct", round(noise_pct * 100, 2))
        trial.set_user_attr("coherence", round(coherence, 4))
        trial.set_user_attr("stability", round(stability, 4))
        trial.set_user_attr("median_cluster_size", round(median_size, 1))
        trial.set_user_attr("score", round(score, 4))

        return score

    return objective

## 6. Full Optuna search

In [7]:
N_TRIALS = 200

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20),
)

objective = make_objective(embeddings_pca, embeddings)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest score: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
best_attrs = study.best_trial.user_attrs
print(f"  n_clusters: {best_attrs.get('n_clusters')}")
print(f"  noise_pct:  {best_attrs.get('noise_pct')}%")
print(f"  coherence:  {best_attrs.get('coherence')}")
print(f"  stability:  {best_attrs.get('stability')}")

# save params immediately
import json as _json
with open("best_params.json", "w") as f:
    _json.dump({"best_params": study.best_params,
                "best_value": study.best_value,
                "best_attrs": best_attrs}, f, indent=2)
print("\nSaved best_params.json")

Best trial: 128. Best value: 0.808067: 100%|██████████| 200/200 [55:04<00:00, 16.52s/it]


Best score: 0.8081
Best params: {'n_neighbors': 11, 'n_components': 13, 'min_dist': 0.014138059874308344, 'min_cluster_size': 32, 'min_samples': 4}
  n_clusters: 116
  noise_pct:  15.45%
  coherence:  0.8126
  stability:  0.0892

Saved best_params.json


## 7. Fit the best model

In [8]:
bp = study.best_params

reducer_final = umap.UMAP(
    n_neighbors=bp["n_neighbors"],
    n_components=bp["n_components"],
    min_dist=bp["min_dist"],
    metric="cosine",
    random_state=42,
)
X_umap = reducer_final.fit_transform(embeddings_pca)

clusterer_final = hdbscan.HDBSCAN(
    min_cluster_size=bp["min_cluster_size"],
    min_samples=bp["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
)
labels = clusterer_final.fit_predict(X_umap)

n_clusters = len(set(labels) - {-1})
noise_pct = float(np.mean(labels == -1)) * 100
coherence = semantic_coherence(embeddings, labels)

print(f"Final model:")
print(f"  Clusters: {n_clusters}")
print(f"  Noise: {noise_pct:.1f}%")
print(f"  Semantic coherence: {coherence:.4f}")

df["cluster_label"] = labels

Final model:
  Clusters: 116
  Noise: 15.5%
  Semantic coherence: 0.8126


## 8. Cluster merging

In [9]:
MERGE_THRESHOLD = 0.92


def merge_similar_clusters(embeddings, labels, threshold):
    cluster_ids = sorted(set(labels) - {-1})
    if not cluster_ids:
        return {}

    centroids = {c: embeddings[labels == c].mean(axis=0) for c in cluster_ids}
    centroid_matrix = np.array([centroids[c] for c in cluster_ids])
    sims = cosine_similarity(centroid_matrix)

    # build adjacency
    n = len(cluster_ids)
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            if sims[i, j] > threshold:
                adj[i].add(j)
                adj[j].add(i)

    # connected components via BFS
    visited = set()
    components = []
    for start in range(n):
        if start in visited:
            continue
        component = []
        queue = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.append(cluster_ids[node])
            queue.extend(adj[node] - visited)
        components.append(component)

    return {comp[0]: comp for comp in components}


def relabel(labels, merged_groups):
    mapping = {}
    for new_id, (_, group) in enumerate(merged_groups.items()):
        for old_id in group:
            mapping[old_id] = new_id
    return np.array([mapping.get(l, -1) for l in labels])


groups = merge_similar_clusters(embeddings, labels, MERGE_THRESHOLD)
merged_labels = relabel(labels, groups)

n_before = len(set(labels) - {-1})
n_after = len(set(merged_labels) - {-1})
print(f"Clusters before merge: {n_before}")
print(f"Clusters after merge:  {n_after}")
print(f"Merged away: {n_before - n_after}")

df["merged_label"] = merged_labels

Clusters before merge: 116
Clusters after merge:  107
Merged away: 9


## 9. Noise reassignment

SIM_THRESHOLD = 0.80, VAR_THRESHOLD = 0.25, AMBIGUITY_MARGIN = 0.05.

In [10]:
SIM_THRESHOLD = 0.80
VAR_THRESHOLD = 0.25
AMBIGUITY_MARGIN = 0.05

cluster_ids = sorted(set(merged_labels) - {-1})
centroids, variances = {}, {}

for c in cluster_ids:
    vecs = embeddings[merged_labels == c]
    centroid = vecs.mean(axis=0)
    sims = cosine_similarity(vecs, centroid.reshape(1, -1)).ravel()
    centroids[c] = centroid
    variances[c] = float(1 - np.mean(sims))

# eligible clusters
eligible = [c for c in cluster_ids if variances[c] <= VAR_THRESHOLD]
print(f"Eligible clusters for reassignment: {len(eligible)} / {len(cluster_ids)}")
print(f"(blocked by variance threshold: {len(cluster_ids) - len(eligible)})")

noise_idx = np.where(merged_labels == -1)[0]
final_labels = merged_labels.copy()
reassigned = 0
confidence = {}

if eligible:
    eligible_centroids = np.array([centroids[c] for c in eligible])

    for idx in noise_idx:
        vec = embeddings[idx].reshape(1, -1)
        sims = cosine_similarity(vec, eligible_centroids).ravel()

        best_pos = int(np.argmax(sims))
        best_sim = float(sims[best_pos])

        if best_sim < SIM_THRESHOLD:
            continue

        # ambiguity check
        if len(sims) > 1:
            second_best = float(np.sort(sims)[-2])
            if best_sim - second_best < AMBIGUITY_MARGIN:
                continue  # genuinely ambiguous, leave as noise

        final_labels[idx] = eligible[best_pos]
        confidence[idx] = round(best_sim, 4)
        reassigned += 1

noise_before = int(np.sum(merged_labels == -1))
noise_after = int(np.sum(final_labels == -1))
print(f"\nNoise before: {noise_before} ({noise_before/len(final_labels)*100:.1f}%)")
print(f"Noise after:  {noise_after} ({noise_after/len(final_labels)*100:.1f}%)")
print(f"Reassigned:   {reassigned} ({reassigned/max(noise_before,1)*100:.1f}% of noise points)")

df["final_label"] = final_labels

Eligible clusters for reassignment: 96 / 107
(blocked by variance threshold: 11)

Noise before: 2107 (15.5%)
Noise after:  2011 (14.7%)
Reassigned:   96 (4.6% of noise points)


## 10. Inspect cluster sizes

In [11]:
label_counts = Counter(final_labels)
n_noise = label_counts.get(-1, 0)
cluster_sizes = sorted([v for k, v in label_counts.items() if k != -1], reverse=True)

print(f"Final clusters: {len(cluster_sizes)}")
print(f"Noise points:   {n_noise} ({n_noise/len(final_labels)*100:.1f}%)")
print(f"Largest 10 clusters: {cluster_sizes[:10]}")
print(f"Smallest 10 clusters: {cluster_sizes[-10:]}")
print(f"Median cluster size: {np.median(cluster_sizes):.0f}")

Final clusters: 107
Noise points:   2011 (14.7%)
Largest 10 clusters: [938, 819, 390, 329, 324, 308, 264, 256, 254, 248]
Smallest 10 clusters: [33, 33, 33, 33, 33, 32, 32, 32, 32, 32]
Median cluster size: 72


## 11. Save

In [12]:
df_out = df.copy()
df_out["to_lists"] = df_out["to_lists"].apply(json.dumps)
df_out["to_other_recipients"] = df_out["to_other_recipients"].apply(json.dumps)

df_out.to_csv("tovima_clustered.csv", sep="\t", encoding="utf-32", index=False)
np.save("umap_projection.npy", X_umap)

print(f"Saved tovima_clustered.csv: {df_out.shape}")
print(f"Saved umap_projection.npy: {X_umap.shape}")
print()
print("Columns in output:")
for col in df_out.columns:
    print(f"  {col}")

Saved tovima_clustered.csv: (13637, 18)
Saved umap_projection.npy: (13637, 13)

Columns in output:
  Author
  Date
  To
  Subject
  Message
  Date_parsed
  Year
  had_html_markup
  is_reply_or_forward
  to_lists
  to_other_recipients
  subject_tag
  contains_pii_pattern
  embedding_text
  cleaned_text
  cluster_label
  merged_label
  final_label
